## Implementation

### Step 1 — Point at the same catalog/schema as Notebooks 02 and 05

In [0]:
dbutils.widgets.text("catalog_name", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema_name", "genai_lab", "Schema")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
documents_table = f"{catalog_name}.{schema_name}.documents"

print(f"Documents table: {documents_table}")

### Step 2 — Load the documents and rebuild the header-plus-body text

`department` and `created_date` are the ground truth we hand-assigned in Notebook 02. They live in their own columns, not inside `content` -- so before extracting anything, we prepend a short metadata header to each document's body, the same shape Notebook 02 wrote to `.txt` files. `ai_extract` will only ever see the combined `document_text` column; the `department` and `created_date` columns are kept aside, untouched, purely as ground truth to check the extraction against in Step 4.

In [0]:
from pyspark.sql.functions import concat_ws, col, lit

documents_df = spark.table(documents_table)

documents_with_text_df = documents_df.withColumn(
    "document_text",
    concat_ws(
        "",
        lit("Title: "), col("title"), lit("\n"),
        lit("Category: "), col("category"), lit("\n"),
        lit("Department: "), col("department"), lit("\n"),
        lit("Created: "), col("created_date"), lit("\n\n"),
        col("content"),
    ),
)
documents_with_text_df.createOrReplaceTempView("documents_v")

# Sanity check: confirm the header actually landed in document_text before extracting from it
print(documents_with_text_df.select("document_text").first()["document_text"])

### Step 3 — Extract a fixed, two-field schema from every document

`ai_extract(text, fields)` takes the text to extract from and an array of field names, and returns a `STRUCT` with one value per field, accessed by dot notation (`.department`, `.effective_date`). We ask for the same two fields on every document regardless of category -- the simplest possible schema, and the one most directly comparable to ground truth.

**Run this cell first and inspect the raw output before trusting the dot-notation access in later cells** -- confirm your workspace actually returns a `STRUCT` shaped the way this notebook assumes.

In [0]:
%sql
SELECT
  doc_id,
  title,
  department AS actual_department,
  created_date AS actual_effective_date,
  ai_extract(
    document_text,
    ARRAY('department', 'effective_date')
  ) AS extracted
FROM documents_v
ORDER BY doc_id

### Step 4 — Capture the result in Python and score each field against ground truth

Same query, kept as a DataFrame so we can pull the `department` and `effective_date` fields out of the `extracted` struct individually and compare each one to its ground-truth column. Note the two fields are scored separately -- a document can get `department` right and `effective_date` wrong, or vice versa, in a way that a single classification accuracy number couldn't express.

In [0]:
extracted_df = spark.sql(
    """
    SELECT
      doc_id,
      title,
      department AS actual_department,
      created_date AS actual_effective_date,
      ai_extract(document_text, ARRAY('department', 'effective_date')) AS extracted
    FROM documents_v
    """
).select(
    "doc_id",
    "title",
    "actual_department",
    "actual_effective_date",
    col("extracted.department").alias("extracted_department"),
    col("extracted.effective_date").alias("extracted_effective_date")
)

total = extracted_df.count()
department_correct = extracted_df.filter("actual_department = extracted_department").count()
date_correct = extracted_df.filter("actual_effective_date = extracted_effective_date").count()

print(f"department match:      {department_correct}/{total} ({100 * department_correct / total:.1f}%)")
print(f"effective_date match:  {date_correct}/{total} ({100 * date_correct / total:.1f}%)")

print("\ndepartment mismatches (worth reading, not just counting):")
display(extracted_df.filter("actual_department != extracted_department OR extracted_department IS NULL"))

print("\neffective_date mismatches:")
display(extracted_df.filter("actual_effective_date != extracted_effective_date OR extracted_effective_date IS NULL"))

### Step 5 — Extract a field that genuinely is not in the text

Even with the metadata header from Step 2, none of these 15 documents state a regulatory filing number anywhere. Ask for that field and see whether `ai_extract` returns `NULL` (the correct, honest answer) rather than inventing one.

In [0]:
absent_field_df = spark.sql(
    """
    SELECT
      doc_id,
      title,
      ai_extract(document_text, ARRAY('regulatory_filing_number')).regulatory_filing_number AS extracted_filing_number
    FROM documents_v
    """
)

null_count = absent_field_df.filter("extracted_filing_number IS NULL").count()
total = absent_field_df.count()
print(f"NULL (correctly absent): {null_count}/{total}")

print("\nAny non-null value here is worth reading closely -- it means the model found or invented something:")
display(absent_field_df.filter("extracted_filing_number IS NOT NULL"))

### Step 6 — Tailor the schema per category instead of using one fixed schema

`department` and `effective_date` make sense for every document. `product_name` only makes sense for a Product document; `policy_type` only makes sense for a Compliance document. This step asks each category's documents for a field specific to that category, alongside the two universal fields -- a direct look at the one-fixed-schema-for-everything (Step 3) versus schema-per-category trade-off raised in the Conceptual Explanation. Unlike Steps 3-4, these category-specific fields have no ground-truth column to score against -- read the values, don't just count them.

In [0]:
category_fields = {
    "Product": "product_name",
    "Operations": "applicable_process",
    "Compliance": "policy_type",
    "Customer Service": "applicable_process",
    "Technical": "api_resource",
    "Customer": "customer_complaint_id"
}

per_category_results = []
for category, extra_field in category_fields.items():
    rows = spark.sql(
        f"""
        SELECT
          doc_id,
          title,
          category,
          ai_extract(document_text, ARRAY('department', 'effective_date', '{extra_field}')) AS extracted
        FROM documents_v
        WHERE category = '{category}'
        """
    ).select(
        "doc_id",
        "title",
        "category",
        col("extracted.department").alias("department"),
        col("extracted.effective_date").alias("effective_date"),
        col(f"extracted.{extra_field}").alias(f"{extra_field}"),
    )
    per_category_results.append((category, extra_field, rows))

for category, extra_field, rows in per_category_results:
    print(f"\n{category} -- extra field: {extra_field}")
    display(rows)

### Step 7 — Extract from the unlabeled Fraud Escalation Playbook

This document has no metadata header and no `department` or `created_date` ground truth at all -- a genuine test of whether `ai_extract` returns `NULL` for a field that is truly absent from the source text, on a document this notebook has no prior expectation for.

In [0]:
fraud_escalation_text = (
    "Fraud Escalation Playbook\n\n"
    "This playbook defines how Aurora Trust Bank staff escalate suspected fraud cases. "
    "Any transaction flagged by fraud monitoring must be reviewed within one business hour. "
    "Confirmed fraud cases are escalated to the Fraud Operations team, who freeze the affected "
    "account and notify the customer through the contact center. "
    "Escalation Steps: (1) Analyst reviews the flagged transaction. (2) Analyst confirms or "
    "dismisses the fraud indicator. (3) Confirmed cases are routed to Fraud Operations. "
    "(4) Fraud Operations freezes the account and opens a case file. "
    "Escalation Tiers: Tier 1 handles amounts under 1,000 units; Tier 2 handles 1,000-10,000 "
    "units and requires a supervisor sign-off; Tier 3 handles amounts above 10,000 units and "
    "requires notifying the Compliance department."
)

fraud_df = spark.createDataFrame([("fraud_escalation_playbook", fraud_escalation_text)], ["title", "document_text"])
fraud_df.createOrReplaceTempView("fraud_doc_v")

(
    spark.sql(
        """
        SELECT
          title,
          document_text,
          ai_extract(
            document_text,
            ARRAY('department', 'effective_date', 'escalation_tier_count')
          ) AS extracted
        FROM fraud_doc_v
        """
    )
    .select(
        "title",
        "document_text",
        col("extracted.department").alias("department"),
        col("extracted.effective_date").alias("effective_date"),
        col(f"extracted.escalation_tier_count").alias(f"escalation_tier_count"),
    )
).display()